# End of week 1 exercise

To demonstrate your familiarity with the OpenAI-compatible API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

This version uses **Gemini** (via Google's OpenAI-compatible endpoint) with streaming, then **Llama 3.2** locally via Ollama.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GEMINI = "gemini-3.1-flash-lite"
MODEL_LLAMA = "llama3.2"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No Gemini API key found. Add GOOGLE_API_KEY or GEMINI_API_KEY to .env and save the file.")
elif not api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("Gemini API key found and looks good so far!")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
# here is the question; type over this to ask something new

question = """
What is RAG (retrieval-augmented generation) at a high level, and why does it exist?
"""

system_prompt = (
    "You are a helpful technical tutor who answers questions about python code, "
    "software engineering, data science and LLMs. Explain clearly, with structure, "
    "and use markdown."
)
user_prompt = "Please give a detailed explanation to the following question: " + question

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
# Get Gemini to answer, with streaming

stream = gemini.chat.completions.create(
    model=MODEL_GEMINI,
    messages=messages,
    stream=True,
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    response = response.replace("```", "").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# Get Llama 3.2 to answer (local Ollama)

response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
display(Markdown(response.choices[0].message.content))

## Try another question

Change the `question` string above, re-run from that cell, and compare Gemini vs Llama.

If Ollama is slow, pull a smaller model and switch the constant:

```python
MODEL_LLAMA = "llama3.2:1b"
```

If Gemini rejects the model name, try `gemini-2.0-flash` or `gemini-2.5-flash`.